# 🤳 Teaching AI YOUR Face: DreamBooth, LoRA & Personalization

**Chapter 10 — GenAI Design Study Guide**

---

> *How does a generative model go from "draw a person" to "draw THAT specific person"?*

In this notebook we cover the three main personalization techniques used in production headshot systems:

| Technique | Year | Core Idea |
|---|---|---|
| **Textual Inversion** | 2022 | Learn a new text token for your subject |
| **DreamBooth** | 2022 | Fine-tune the whole model with a rare identifier |
| **LoRA** | 2021/2023 | Inject tiny trainable weight updates |

By the end you will be able to:
- Explain the trade-offs between all three methods at a staff-engineer level 🎯
- Implement the core math from scratch in PyTorch 🔥
- Discuss evaluation metrics with confidence 📊

## 🎨 What Does "Personalization" Even Mean?

Imagine you hire an **art teacher** who can paint literally anything — portraits, landscapes, dragons, you name it. They have spent years mastering the craft. Now you say:

> "Great — but I want you to paint *ME*. Specifically me. Not 'a person with brown hair'. Me."

That's the personalization problem. The model already knows *how* to draw faces. It just needs to learn *your* face.

### 🧩 The Hard Part: Only 10–20 Reference Photos

A standard diffusion model trains on **billions** of images. You're giving it **10 to 20** photos of yourself. That's like asking your art teacher to study 15 selfies and then recreate you perfectly in any style, any pose, any lighting. 😬

This creates two core tension points:

1. **Underfitting** — Too little signal → the model forgets who you are and draws a generic face.
2. **Overfitting** — Too much fine-tuning → the model memorizes your 15 photos and loses the ability to put you in new contexts (it can only draw you exactly as it saw you). This is called **language drift** or **catastrophic forgetting**.

### 📐 Formal Problem Statement

Given:
- A pre-trained diffusion model $\epsilon_\theta$
- A small reference set $\mathcal{D} = \{x_1, \ldots, x_N\}$ where $N \approx 10$–$20$

Find a way to generate new images of the **same identity** conditioned on novel text prompts like:
- "[subject] as an astronaut on the moon"
- "[subject] in a renaissance oil painting"
- "[subject] wearing a superhero cape"

Without losing the model's general generative capability.

## 🗺️ Three Roads to Personalization

Here's your high-level map before we go deep:

```
Pre-trained model
      │
      ├──► Textual Inversion  →  only the text embedding changes
      │                          everything else stays frozen
      │
      ├──► DreamBooth         →  the whole model gets fine-tuned
      │                          with a clever rare-token trick
      │
      └──► LoRA               →  tiny side-car matrices injected
                                 into the model's attention layers
```

### Summary Table

| | Textual Inversion | DreamBooth | LoRA |
|---|---|---|---|
| **What's trained?** | One embedding vector (~768 floats) | Entire UNet + text encoder | Small A, B matrices per attention layer |
| **Storage cost** | ~5 KB | 2–6 GB | 5–150 MB |
| **Identity fidelity** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Training time (A100)** | ~1 hour | 3–6 hours | 30–60 min |
| **Forgetting risk** | Very low | High (needs PPL) | Low–Medium |
| **Composability** | Great (mix tokens) | Poor | Good |
| **Production use** | Rare (too weak) | LinkedIn, startup apps | ✅ Industry standard |

> **PPL** = Prior Preservation Loss — DreamBooth's answer to forgetting. We'll build it from scratch below! 🔨

## 🔤 Textual Inversion: Learning a New "Word" for Your Face

**Paper**: *An Image is Worth One Word* (Gal et al., 2022)

### The Big Idea

A text encoder (like CLIP) maps words to **embedding vectors** — points in a high-dimensional space. "Dog" maps to some vector. "Red" maps to another. These embeddings are what the diffusion model uses to understand your prompt.

Textual Inversion asks: *What if we found a new embedding vector $v^*$ that represents your face, and just used that as a pseudo-word $S^*$?*

```
Prompt:  "A photo of S* at a birthday party"
                        ↑
              This is a learned vector,
              not a real word!
```

### What Gets Trained vs Frozen

```
Text Encoder    → ❄️ FROZEN
UNet (denoiser) → ❄️ FROZEN  
VAE (encoder)   → ❄️ FROZEN
Embedding v*    → 🔥 TRAINED  ← only this ~768-dimensional vector!
```

### The Loss Function

Standard diffusion denoising loss, but only the embedding gets gradients:

$$\mathcal{L}_{\text{TI}} = \mathbb{E}_{x, \epsilon, t} \left[ \| \epsilon - \epsilon_\theta(x_t, t, c(v^*)) \|^2 \right]$$

Where $c(v^*)$ is the text conditioning built from our new pseudo-word.

### 👍 Pros
- Almost zero storage (just one vector)
- No risk of destroying the model
- Very composable: `S* + "wearing a hat"` works naturally

### 👎 Cons
- A single vector can't capture everything about a face
- Identity fidelity is noticeably weaker than DreamBooth/LoRA
- Requires careful initialization (usually start from "person" or "face" embedding)

In [ ]:
import torch
import torch.nn as nn

# ============================================================
# 🔤 Textual Inversion: Trainable vs Frozen Parameters
# ============================================================
# We'll build a toy model that mimics the structure of a
# text-conditioned diffusion model and show exactly which
# parameters get gradients.

class ToyTextEncoder(nn.Module):
    """Mimics a CLIP-style text encoder."""
    def __init__(self, vocab_size: int = 10000, embed_dim: int = 64):
        super().__init__()
        # Standard token embeddings (the whole vocabulary)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Transformer layers (simplified as a single Linear)
        self.transformer = nn.Linear(embed_dim, embed_dim)
        # Our new pseudo-word embedding S* — this is what TI trains!
        self.learned_embedding = nn.Parameter(
            torch.randn(1, embed_dim)  # shape: (1, embed_dim)
        )

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        embeds = self.token_embedding(token_ids)
        return self.transformer(embeds)


class ToyUNet(nn.Module):
    """Mimics the denoising UNet backbone."""
    def __init__(self, dim: int = 64):
        super().__init__()
        self.down = nn.Linear(dim, dim)
        self.mid  = nn.Linear(dim, dim)
        self.up   = nn.Linear(dim, dim)

    def forward(self, x, conditioning):
        x = self.down(x) + conditioning
        x = self.mid(x)
        return self.up(x)


class ToyDiffusionModel(nn.Module):
    """Full toy diffusion model for demonstration."""
    def __init__(self):
        super().__init__()
        self.text_encoder = ToyTextEncoder()
        self.unet         = ToyUNet()


def apply_textual_inversion_freeze(model: ToyDiffusionModel) -> ToyDiffusionModel:
    """
    Freeze everything EXCEPT the learned_embedding.
    This is the core operation of Textual Inversion.
    """
    # Step 1: Freeze the entire model
    for param in model.parameters():
        param.requires_grad = False

    # Step 2: Un-freeze ONLY the learned pseudo-word embedding
    model.text_encoder.learned_embedding.requires_grad = True

    return model


def print_trainable_params(model: nn.Module, method_name: str):
    """Print a table of trainable vs frozen parameters."""
    print(f"\n{'='*60}")
    print(f"  {method_name} — Parameter Status")
    print(f"{'='*60}")
    total_params     = 0
    trainable_params = 0
    print(f"  {'Parameter':<45} {'Trainable?':>10}")
    print(f"  {'-'*55}")
    for name, param in model.named_parameters():
        total_params += param.numel()
        status = "🔥 YES" if param.requires_grad else "❄️  no"
        if param.requires_grad:
            trainable_params += param.numel()
        print(f"  {name:<45} {status:>10}")
    print(f"  {'-'*55}")
    print(f"  Total params:     {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")
    pct = 100.0 * trainable_params / total_params
    print(f"  % trainable:      {pct:.4f}%")
    print(f"{'='*60}\n")


# --- Before Textual Inversion freeze ---
model = ToyDiffusionModel()
print_trainable_params(model, "Baseline (all trainable)")

# --- After Textual Inversion freeze ---
model = apply_textual_inversion_freeze(model)
print_trainable_params(model, "Textual Inversion")

# --- Sanity check: only learned_embedding has grad ---
assert model.text_encoder.learned_embedding.requires_grad, "learned_embedding should be trainable!"
assert not model.unet.down.weight.requires_grad, "UNet should be frozen!"
print("✅ Sanity check passed — only learned_embedding is trainable.")

## 🎯 DreamBooth: Fine-Tune the Whole Model (Carefully!)

**Paper**: *DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation* (Ruiz et al., 2022)

### The Big Idea

Instead of only touching the text embedding, DreamBooth says: *fine-tune the **entire** diffusion model* on your photos. But to tell the model "this is YOUR face specifically", we give you a special identifier:

```
Training prompt: "A photo of [V] person"
Inference:       "[V] person as an astronaut"
                              ↑
               [V] is a RARE TOKEN — e.g., "sks"
```

### Why Rare Tokens? The Goldilocks Problem 🐻

The identifier token needs to be **just right**:

| Token type | Example | Problem |
|---|---|---|
| **Too common** | "john" | Strong prior associations → model fights to keep John generic |
| **Too random (multi-token)** | "xqzjbt" | Tokenizer splits it into multiple tokens → messy conditioning |
| **Just right (rare, single-token)** | "sks", "zwx", "ktn" | Exists in vocabulary, almost no prior → blank slate for your face |

### Why Common Words Fail 🚫

If you use `[john]` as your identifier, the model has seen millions of "John" images during pretraining. When you fine-tune, the model experiences a **tug of war**:
- New gradient: "John = this specific person's face"
- Prior gradient: "John = any generic man named John"

The rare token "sks" has no such baggage — the model only knows about "sks" from **your** fine-tuning data.

### Why Random Strings Can Also Fail 🚫

CLIP's tokenizer (BPE) breaks unknown strings into subword pieces:
- `"xqzjbt"` → `["xq", "zj", "bt"]` — now you have 3 tokens, not 1!
- Multi-token identifiers create ambiguous conditioning and harder optimization.

The sweet spot: tokens that **already exist** in the vocabulary as a single token, but appear so rarely they carry no semantic baggage.

In [ ]:
import random
import string
from collections import Counter

# ============================================================
# 🎯 DreamBooth: Rare Token Selection Logic
# ============================================================
# In practice you'd load the actual CLIP/T5 vocabulary and
# corpus frequency counts. Here we simulate that logic.

# Simulated vocabulary with token -> frequency in training corpus
# (higher number = seen more often = stronger prior associations)
SIMULATED_VOCAB = {
    # Very common — terrible identifiers
    "person":    9_800_000,
    "man":       7_200_000,
    "woman":     6_500_000,
    "john":      4_100_000,
    "face":      3_900_000,
    "photo":     8_200_000,
    # Moderately common
    "avatar":      250_000,
    "pixel":       310_000,
    # Rare single tokens — DreamBooth sweet spot ✅
    "sks":              47,
    "zwx":              12,
    "ktn":              23,
    "brn":              31,
    "xpt":               8,
    # Not in vocab (would be split by BPE) — unusable
    # These are NOT in our dict, simulating unknown tokens
}

# Common English words that would be terrible identifiers
TERRIBLE_IDENTIFIERS = ["person", "man", "woman", "john", "face"]

# Candidate tokens we're evaluating
CANDIDATES = ["person", "john", "avatar", "pixel", "sks", "zwx", "ktn", "xqzjbt", "brn"]

FREQ_THRESHOLD_HIGH = 100_000   # Too common — has strong prior
FREQ_THRESHOLD_LOW  = 1         # Too rare — might not be single token


def evaluate_token(token: str, vocab: dict) -> dict:
    """Score a candidate identifier token for DreamBooth suitability."""
    in_vocab   = token in vocab
    frequency  = vocab.get(token, None)  # None means NOT in vocab

    if not in_vocab:
        quality = "❌ NOT IN VOCAB (BPE will split it)"
        score   = 0
    elif frequency > FREQ_THRESHOLD_HIGH:
        quality = "❌ TOO COMMON (strong prior associations)"
        score   = 1
    elif frequency > 500:
        quality = "⚠️  MODERATE (some prior, use with caution)"
        score   = 2
    else:
        quality = "✅ RARE SWEET SPOT (ideal DreamBooth token!)"
        score   = 3

    return {
        "token":     token,
        "in_vocab":  in_vocab,
        "frequency": frequency,
        "quality":   quality,
        "score":     score,
    }


print("DreamBooth Rare Token Selection Analysis")
print("=" * 65)
print(f"  {'Token':<12} {'In Vocab':>9} {'Frequency':>12}  Quality")
print(f"  {'-'*61}")

best_tokens = []
for token in CANDIDATES:
    result = evaluate_token(token, SIMULATED_VOCAB)
    freq_str = f"{result['frequency']:,}" if result['frequency'] is not None else "N/A"
    print(f"  {result['token']:<12} {str(result['in_vocab']):>9} {freq_str:>12}  {result['quality']}")
    if result["score"] == 3:
        best_tokens.append(token)

print(f"\n  Best identifier candidates: {best_tokens}")
print(f"  → DreamBooth would pick: '{best_tokens[0]}' as [V]")
print(f"\n  Training prompts would look like:")
print(f"    'A photo of {best_tokens[0]} person'")
print(f"    '{best_tokens[0]} person smiling'")
print(f"  Inference prompts:")
print(f"    '{best_tokens[0]} person as an astronaut on the moon'")
print(f"    '{best_tokens[0]} person in a renaissance oil painting'")

## 🧠 Prior Preservation Loss: Don't Forget Other Faces!

Here's the catastrophic forgetting problem that DreamBooth has to solve:

```
Before fine-tuning:  "a person" → draws diverse faces 👨👩👴👧
After naive fine-tuning: "a person" → draws YOUR face every time 😱
```

Why? Because you fine-tune on images of YOU, and the model updates its weights so strongly that "person" becomes associated with your specific identity. The model **forgets** what generic people look like.

### The Fix: Class-Specific Prior Preservation Loss (PPL)

DreamBooth's solution is elegant: **simultaneously train on generated images of OTHER people**.

```
Batch during training:
  ┌─────────────────────────────────────────────────────────┐
  │  Subject images:  "A photo of sks person"  (YOUR face)  │
  │  Prior images:    "A photo of a person"    (ANY face)   │  ← generated by the model itself before fine-tuning
  └─────────────────────────────────────────────────────────┘
```

The prior images act as **anchors** — they remind the model what generic faces look like and prevent it from drifting.

### The Math

$$\mathcal{L}_{\text{total}} = \underbrace{\alpha \cdot \mathbb{E}\left[ \| \epsilon - \epsilon_\theta(x_t^{\text{subj}}, t, c_{\text{subj}}) \|^2 \right]}_{\mathcal{L}_{\text{reconstruction}} — \text{learn the subject}} + \underbrace{\beta \cdot \mathbb{E}\left[ \| \epsilon - \epsilon_\theta(x_t^{\text{prior}}, t, c_{\text{prior}}) \|^2 \right]}_{\mathcal{L}_{\text{prior}} — \text{don't forget the class}}$$

Where:
- $x^{\text{subj}}$ = your reference photos
- $x^{\text{prior}}$ = images generated by the **original** model for "a person"
- $\alpha, \beta$ are weighting hyperparameters (typically $\alpha = 1.0$, $\beta = 1.0$)

The prior images are generated **once before training** and stored. They come from the model itself — so they represent "what the model thinks a person looks like" before it starts learning your face.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 🧠 DreamBooth Prior Preservation Loss — From Scratch
# ============================================================

def reconstruction_loss(
    predicted_noise: torch.Tensor,
    actual_noise:    torch.Tensor,
) -> torch.Tensor:
    """
    L_reconstruction: how well we predict the noise added to subject images.
    A low value means the model is learning the subject's identity.
    """
    return F.mse_loss(predicted_noise, actual_noise)


def prior_preservation_loss(
    predicted_noise_generic: torch.Tensor,
    actual_noise_generic:    torch.Tensor,
) -> torch.Tensor:
    """
    L_prior: how well we predict noise for generic class images.
    A low value means the model remembers how to draw generic faces.
    """
    return F.mse_loss(predicted_noise_generic, actual_noise_generic)


def dreambooth_total_loss(
    pred_subject:  torch.Tensor,
    true_subject:  torch.Tensor,
    pred_prior:    torch.Tensor,
    true_prior:    torch.Tensor,
    alpha: float = 1.0,
    beta:  float = 1.0,
) -> dict:
    """
    DreamBooth's full training objective.
    Returns a dict with all loss components for visibility.
    """
    l_recon = reconstruction_loss(pred_subject, true_subject)
    l_prior = prior_preservation_loss(pred_prior, true_prior)
    l_total = alpha * l_recon + beta * l_prior
    return {
        "L_reconstruction": l_recon.item(),
        "L_prior":           l_prior.item(),
        "L_total":           l_total.item(),
        "alpha":             alpha,
        "beta":              beta,
    }


# --- Simulate training dynamics over 500 steps ---
torch.manual_seed(42)
steps = 500

# We'll simulate how these losses evolve.
# Subject loss drops fast (small dataset, easy to overfit).
# Prior loss stays low because we have many prior images.
history = {
    "L_reconstruction": [],
    "L_prior_with_ppl": [],
    "L_prior_no_ppl":   [],
    "L_total":          [],
}

for step in range(steps):
    t = step / steps  # normalized time 0→1

    # Simulate noise tensors (batch=4, latent_dim=16)
    true_subj  = torch.randn(4, 16)
    true_prior = torch.randn(4, 16)

    # Subject prediction improves with training (noisy but converging)
    subj_noise = max(0.05, 1.0 * (1 - t) ** 2.5 + 0.05 * torch.randn(1).item())
    pred_subj  = true_subj + subj_noise * torch.randn_like(true_subj)

    # Prior WITH PPL: stays well-calibrated
    prior_noise_good = max(0.1, 0.2 + 0.05 * torch.randn(1).item())
    pred_prior_good  = true_prior + prior_noise_good * torch.randn_like(true_prior)

    # Prior WITHOUT PPL: drifts as model forgets generic faces
    prior_noise_bad = min(2.0, 0.2 + t * 1.8 + 0.1 * torch.randn(1).item())
    pred_prior_bad  = true_prior + prior_noise_bad * torch.randn_like(true_prior)

    losses = dreambooth_total_loss(pred_subj, true_subj, pred_prior_good, true_prior)

    history["L_reconstruction"].append(losses["L_reconstruction"])
    history["L_prior_with_ppl"].append(F.mse_loss(pred_prior_good, true_prior).item())
    history["L_prior_no_ppl"].append(F.mse_loss(pred_prior_bad, true_prior).item())
    history["L_total"].append(losses["L_total"])


# Smooth with rolling mean for cleaner plot
def smooth(values, window=20):
    return np.convolve(values, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("DreamBooth Training Dynamics", fontsize=14, fontweight='bold')

# Left: Loss components
ax = axes[0]
ax.plot(smooth(history["L_reconstruction"]), label="L_reconstruction (subject)",  color="royalblue",   lw=2)
ax.plot(smooth(history["L_total"]),          label="L_total (with PPL)",           color="darkorange",  lw=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Components")
ax.legend()
ax.grid(alpha=0.3)

# Right: Prior drift comparison
ax = axes[1]
ax.plot(smooth(history["L_prior_with_ppl"]), label="Prior loss WITH PPL",     color="green",     lw=2)
ax.plot(smooth(history["L_prior_no_ppl"]),   label="Prior loss WITHOUT PPL",  color="crimson",   lw=2, linestyle='--')
ax.axhline(y=0.25, color='gray', linestyle=':', alpha=0.7, label='Acceptable threshold')
ax.set_xlabel("Training Step")
ax.set_ylabel("Prior Loss (higher = more forgetting)")
ax.set_title("Prior Preservation: With vs Without PPL")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("dreambooth_training_dynamics.png", dpi=100, bbox_inches='tight')
plt.show()

print("\nFinal loss values (step 500):")
print(f"  L_reconstruction:  {history['L_reconstruction'][-1]:.4f}")
print(f"  L_prior (w/ PPL):  {history['L_prior_with_ppl'][-1]:.4f}")
print(f"  L_prior (no PPL):  {history['L_prior_no_ppl'][-1]:.4f}")
print(f"\n  Without PPL, the prior loss is ~{history['L_prior_no_ppl'][-1]/history['L_prior_with_ppl'][-1]:.1f}x worse — catastrophic forgetting!")

## ⚡ LoRA: The Efficient Approach

**Paper**: *LoRA: Low-Rank Adaptation of Large Language Models* (Hu et al., 2021), popularized for diffusion models in 2023.

### The Big Idea

DreamBooth fine-tunes billions of parameters. LoRA says: *What if the update $\Delta W$ doesn't need to be full rank?*

For a weight matrix $W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, instead of training $\Delta W$ directly:

$$W_{\text{new}} = W_{\text{frozen}} + \Delta W = W_{\text{frozen}} + \underbrace{B}_{d_{\text{out}} \times r} \cdot \underbrace{A}_{r \times d_{\text{in}}}$$

Where $r \ll d$ is the **rank** (typically 4, 8, or 16).

```
Original weight matrix W:
  Shape: [768, 768] = 589,824 parameters  ← FROZEN

LoRA decomposition (r=4):
  A: [4,   768] =   3,072 parameters  ← TRAINED
  B: [768,   4] =   3,072 parameters  ← TRAINED
  Total: 6,144 parameters  =  1% of original!
```

### 🏗️ Where Are LoRA Adapters Injected?

In diffusion models, LoRA is added to the **attention layers** of the UNet:
- Query (Q), Key (K), Value (V) projections
- Output projections
- Sometimes also the feed-forward layers

### 🎛️ The Scaling Factor α

In practice, LoRA uses a scaling hyperparameter:

$$W_{\text{new}} = W + \frac{\alpha}{r} \cdot BA$$

- Setting $\alpha = r$ gives no scaling (same as $\Delta W = BA$)
- Smaller $\alpha/r$ → more conservative updates
- This makes it easy to control "how much LoRA" to apply at inference time!

### 🏃 Why LoRA Won in Production

1. **Multiple LoRAs**: You can combine them! `lora_face + lora_style + lora_lighting`
2. **Strength control**: Multiply each LoRA by a scalar weight at inference
3. **Storage**: 5–150 MB vs 2–6 GB for DreamBooth
4. **Speed**: Fine-tune in 30 minutes vs hours
5. **Reversible**: The base model stays frozen — just remove the adapters

In [ ]:
import torch
import torch.nn as nn
import math

# ============================================================
# ⚡ LoRA Layer Implementation from Scratch
# ============================================================

class LoRALinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that adds LoRA adapters.

    The original weight W is frozen. We inject two low-rank matrices:
        A: (r, d_in)   — projects down
        B: (d_out, r)  — projects up
    Output = W·x + (alpha/r) · B·(A·x)
    """

    def __init__(
        self,
        d_in:    int,
        d_out:   int,
        rank:    int   = 4,
        alpha:   float = 1.0,
        bias:    bool  = True,
    ):
        super().__init__()
        assert rank <= min(d_in, d_out), f"rank {rank} must be <= min({d_in}, {d_out})"

        self.d_in  = d_in
        self.d_out = d_out
        self.rank  = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # The alpha/r scaling factor

        # --- Original frozen weight ---
        self.linear = nn.Linear(d_in, d_out, bias=bias)
        # Freeze the base layer
        for param in self.linear.parameters():
            param.requires_grad = False

        # --- LoRA trainable matrices ---
        # A is initialized with Kaiming uniform (standard)
        # B is initialized to ZERO — so at start, ΔW = BA = 0
        # This means LoRA starts as an identity transform: no change!
        self.lora_A = nn.Parameter(torch.empty(rank, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))  # zeros!
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Original frozen output
        base_output = self.linear(x)
        # LoRA delta: x -> A -> B, scaled by alpha/r
        lora_output = (x @ self.lora_A.T) @ self.lora_B.T
        return base_output + self.scaling * lora_output

    def count_params(self) -> dict:
        """Return parameter counts for original vs LoRA."""
        original = self.d_in * self.d_out
        lora     = self.rank * (self.d_in + self.d_out)
        return {
            "original_params": original,
            "lora_params":     lora,
            "reduction_pct":   100 * (1 - lora / original),
            "trainable_params": lora,
            "frozen_params":   original,
        }


# --- Demo: a typical attention projection in SD (d=768) ---
D = 768  # hidden dimension in Stable Diffusion

print("LoRA Layer Demo (d_in = d_out = 768)")
print("=" * 50)

for r in [1, 2, 4, 8, 16, 32, 64]:
    layer  = LoRALinear(D, D, rank=r)
    counts = layer.count_params()
    print(
        f"  rank={r:>2}  "
        f"Original: {counts['original_params']:>7,}  "
        f"LoRA: {counts['lora_params']:>6,}  "
        f"Savings: {counts['reduction_pct']:>5.1f}%"
    )

# --- Verify that base weights are truly frozen ---
lora_layer = LoRALinear(D, D, rank=4)
trainable  = [(n, p.shape) for n, p in lora_layer.named_parameters() if p.requires_grad]
frozen     = [(n, p.shape) for n, p in lora_layer.named_parameters() if not p.requires_grad]

print("\nTrainable parameters:")
for name, shape in trainable:
    print(f"  🔥 {name}: {shape}")

print("\nFrozen parameters:")
for name, shape in frozen:
    print(f"  ❄️  {name}: {shape}")

# --- Verify B=0 initialization means zero delta at start ---
x    = torch.randn(1, D)
base = lora_layer.linear(x)
full = lora_layer(x)
delta_norm = (full - base).norm().item()
print(f"\n  Delta at initialization (should be ~0): {delta_norm:.8f}")
assert delta_norm < 1e-6, "LoRA delta should be zero at init!"
print("  ✅ B=0 initialization confirmed — LoRA starts as identity!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 📊 Visualize LoRA Parameter Savings
# ============================================================

ranks    = [1, 2, 4, 8, 16, 32, 64, 128]
d        = 768

original_params = d * d
lora_params     = [r * (d + d) for r in ranks]
savings_pct     = [100 * (1 - lp / original_params) for lp in lora_params]
param_counts    = [lp for lp in lora_params]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("LoRA Parameter Efficiency (d_in = d_out = 768)", fontsize=13, fontweight='bold')

# Left: Raw parameter counts
ax = axes[0]
bars = ax.bar(
    [f"r={r}" for r in ranks],
    param_counts,
    color=plt.cm.Blues(np.linspace(0.3, 0.9, len(ranks))),
    edgecolor='navy',
    linewidth=0.8,
)
ax.axhline(y=original_params, color='crimson', linestyle='--', lw=2,
           label=f'Original: {original_params:,}')
for bar, count in zip(bars, param_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{count:,}', ha='center', va='bottom', fontsize=8)
ax.set_xlabel("LoRA Rank")
ax.set_ylabel("Number of Parameters")
ax.set_title("LoRA Params vs Original")
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Right: Savings percentage
ax = axes[1]
colors = ['#2ecc71' if s > 95 else '#f39c12' if s > 80 else '#e74c3c' for s in savings_pct]
bars2 = ax.bar(
    [f"r={r}" for r in ranks],
    savings_pct,
    color=colors,
    edgecolor='black',
    linewidth=0.8,
)
ax.axhline(y=99.0, color='green',   linestyle=':', alpha=0.7, label='99% savings line')
ax.axhline(y=95.0, color='orange',  linestyle=':', alpha=0.7, label='95% savings line')
for bar, pct in zip(bars2, savings_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 3,
            f'{pct:.1f}%', ha='center', va='top', fontsize=8, color='white', fontweight='bold')
ax.set_xlabel("LoRA Rank")
ax.set_ylabel("Parameter Savings (%)")
ax.set_title("% Parameters Saved vs Full Fine-tuning")
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("lora_parameter_savings.png", dpi=100, bbox_inches='tight')
plt.show()

print("\nKey insight table:")
print(f"  {'Rank':<6} {'LoRA Params':>12} {'Savings':>10} {'Use case'}")
print(f"  {'-'*55}")
use_cases = {
    1:   "Minimal style shift",
    2:   "Very lightweight",
    4:   "✅ Production sweet spot",
    8:   "✅ High quality balance",
    16:  "Maximum quality",
    32:  "Near full fine-tune",
    64:  "Approaching DreamBooth",
    128: "Effectively full FT",
}
for r, lp, s in zip(ranks, param_counts, savings_pct):
    print(f"  {r:<6} {lp:>12,} {s:>9.1f}%  {use_cases[r]}")

## 🏆 Comparison: Textual Inversion vs DreamBooth vs LoRA

Now that we've implemented all three, here's the definitive comparison table:

| Criterion | Textual Inversion | DreamBooth | LoRA |
|---|---|---|---|
| **Identity Fidelity** | ⭐⭐ Low | ⭐⭐⭐⭐⭐ Highest | ⭐⭐⭐⭐ High |
| **Training Speed** | ~1 hr | 3–6 hrs | 30–60 min |
| **Storage per Subject** | ~5 KB | 2–6 GB | 5–150 MB |
| **Catastrophic Forgetting Risk** | Very Low | High (needs PPL) | Low |
| **Composability** | Excellent | Poor | Good |
| **Inference Cost** | Normal | Normal | Normal + tiny overhead |
| **Hyperparameter Sensitivity** | Low | High | Medium |
| **Multiple subjects** | Stack tokens | Separate models | Stack adapters |
| **Open source ecosystem** | Limited | Moderate | Massive (HuggingFace) |
| **Industry adoption (2024)** | Research only | Some startups | ✅ Dominant |

### 🎯 When to Use What

```
Textual Inversion → Quick experiments, minimal compute, need composability
DreamBooth        → Maximum quality, don't care about model size, offline use
LoRA              → Production systems, serving many users, cost-sensitive
```

### 🏭 Real-World Production Reality

Most production headshot systems (LinkedIn AI Photo, Aragon AI, Astria, etc.) use **LoRA with rank 4–16** because:

1. You can store millions of user LoRAs without running out of disk space
2. Training per-user takes only 30 min on 1 GPU — scales economically
3. LoRA adapters can be merged or swapped at inference time
4. Strong HuggingFace ecosystem support (PEFT library)

## 📏 Evaluation Metrics: How Do You Know It Works?

You've trained your personalization model. Now how do you measure whether it actually looks like you?

### Four Key Metrics

#### 1. CLIPScore — Text Alignment
**"Does the image match the text prompt?"**

- Compute cosine similarity between the CLIP image embedding and CLIP text embedding
- Range: −1 to 1 (higher = better text-image alignment)
- Problem: CLIP doesn't care about *which specific person* is in the image — only that it looks like "a person at the beach"

$$\text{CLIPScore} = \cos(\text{CLIP\_img}(I), \text{CLIP\_txt}(T))$$

#### 2. FID (Fréchet Inception Distance) — Image Quality
**"Do generated images look like real photos?"**

- Compare feature distributions of generated images vs real photos using Inception-v3
- Lower FID = more realistic images
- Limitation: FID measures population-level quality, not individual identity

$$\text{FID} = \| \mu_r - \mu_g \|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$

#### 3. DINO Score — Visual Similarity
**"Do the generated images visually match the reference photos?"**

- Use DINO (self-supervised ViT) embeddings of generated vs reference images
- DINO captures fine visual details (texture, shape, specific features)
- Higher cosine similarity = better visual match

#### 4. Facial Similarity Score — Identity Preservation ⭐ Most Important
**"Is this actually the same person?"**

- Use a face recognition model (ArcFace, FaceNet, InsightFace) to extract face embeddings
- Compute cosine similarity between generated face and reference face embeddings
- This is the metric that matters most for headshot applications

$$\text{FaceSim}(I_{\text{gen}}, I_{\text{ref}}) = \frac{f(I_{\text{gen}}) \cdot f(I_{\text{ref}})}{\| f(I_{\text{gen}}) \| \| f(I_{\text{ref}}) \|}$$

Where $f(\cdot)$ is the face recognition feature extractor.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 📏 Facial Similarity Score — Core Concept
# ============================================================
# In production you'd use ArcFace / InsightFace / FaceNet.
# Here we demonstrate the core cosine similarity logic
# and show how to interpret the scores.

def cosine_similarity(v1: torch.Tensor, v2: torch.Tensor) -> float:
    """Compute cosine similarity between two face embedding vectors."""
    v1_norm = F.normalize(v1, dim=-1)
    v2_norm = F.normalize(v2, dim=-1)
    return (v1_norm * v2_norm).sum(dim=-1).item()


def facial_similarity_score(
    generated_face_embedding:  torch.Tensor,
    reference_face_embeddings: list,
) -> dict:
    """
    Compute identity preservation score.
    In practice we compare against ALL reference photos and average.
    """
    similarities = [
        cosine_similarity(generated_face_embedding, ref)
        for ref in reference_face_embeddings
    ]
    return {
        "mean_similarity":  np.mean(similarities),
        "max_similarity":   np.max(similarities),
        "min_similarity":   np.min(similarities),
        "all_similarities": similarities,
    }


# Interpretation thresholds (from ArcFace paper benchmarks)
def interpret_score(score: float) -> str:
    if score >= 0.70:
        return "✅ Excellent — Same person, high confidence"
    elif score >= 0.50:
        return "✅ Good — Same person, reasonable confidence"
    elif score >= 0.30:
        return "⚠️  Uncertain — Possibly same person"
    elif score >= 0.10:
        return "❌ Poor — Different person likely"
    else:
        return "❌ Failed — Completely different"


torch.manual_seed(123)
EMBED_DIM = 512  # ArcFace uses 512-dim embeddings

# Simulate reference face embeddings (10 reference photos of the same person)
# We create a "true" face direction and add small noise for each photo
true_face_direction = F.normalize(torch.randn(EMBED_DIM), dim=0)
reference_embeddings = [
    F.normalize(true_face_direction + 0.15 * torch.randn(EMBED_DIM), dim=0)
    for _ in range(10)
]

# Simulate different quality generation scenarios
scenarios = {}
scenarios["Excellent (LoRA r=16 well-trained)"] = F.normalize(true_face_direction + 0.10 * torch.randn(EMBED_DIM), dim=0)
scenarios["Good (LoRA r=4)"]                    = F.normalize(true_face_direction + 0.40 * torch.randn(EMBED_DIM), dim=0)
scenarios["Mediocre (Textual Inversion)"]        = F.normalize(true_face_direction + 0.90 * torch.randn(EMBED_DIM), dim=0)
scenarios["Poor (wrong person)"]                 = F.normalize(torch.randn(EMBED_DIM), dim=0)

print("Facial Similarity Score Evaluation")
print("=" * 65)
print(f"  Reference photos: 10  |  Embedding dim: {EMBED_DIM}")
print(f"  {'Scenario':<38} {'Mean Sim':>9} {'Verdict'}")
print(f"  {'-'*63}")

scenario_names = []
mean_scores    = []

for name, gen_embed in scenarios.items():
    result = facial_similarity_score(gen_embed, reference_embeddings)
    verdict = interpret_score(result["mean_similarity"])
    print(f"  {name:<38} {result['mean_similarity']:>9.4f}  {verdict}")
    scenario_names.append(name.split('(')[0].strip())
    mean_scores.append(result["mean_similarity"])

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#2ecc71' if s >= 0.5 else '#f39c12' if s >= 0.3 else '#e74c3c' for s in mean_scores]
bars = ax.barh(scenario_names, mean_scores, color=bar_colors, edgecolor='black', linewidth=0.8)
ax.axvline(x=0.50, color='green',  linestyle='--', alpha=0.7, label='Acceptable threshold (0.50)')
ax.axvline(x=0.70, color='blue',   linestyle='--', alpha=0.7, label='Excellent threshold (0.70)')
for bar, score in zip(bars, mean_scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=10)
ax.set_xlabel("Cosine Similarity with Reference Faces")
ax.set_title("Facial Identity Preservation Score by Method")
ax.set_xlim(0, 1.0)
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("facial_similarity_scores.png", dpi=100, bbox_inches='tight')
plt.show()

## 🔭 DINO vs CLIP: Why You Need Both

A common interview question: *"Why use DINO score AND CLIPScore? Aren't they both just measuring similarity?"*

They measure **fundamentally different things**:

### 🎭 The Two-Jackets Example

Consider two photos of the same person:
- Photo A: Alex in a **red jacket**
- Photo B: Alex in a **blue jacket**

| | CLIP | DINO |
|---|---|---|
| **Training objective** | Match images to text descriptions | Self-supervised (no text) |
| **What it captures** | Semantic meaning, concepts, categories | Fine visual structure, texture, shape |
| **Two jackets example** | High similarity ("person in jacket") | Lower similarity (different colors/textures) |
| **Same face, different pose** | High (both are "person") | Medium (pose changes local features) |
| **Good at detecting** | Whether the scene/activity matches | Whether the face/body physically matches |

### 🧠 Technical Intuition

**CLIP** sees: "both are humans wearing jackets" → HIGH similarity
- CLIP was trained to map "photo of a person" close to images of people — regardless of which specific person

**DINO** sees: "different jacket color, different texture, slightly different face angle" → LOWER similarity
- DINO learns patch-level visual features without any text supervision
- It notices pixel-level differences that CLIP ignores

### 📐 What Each Metric Actually Measures in Headshot Systems

```
CLIPScore   → "Does the prompt style/scene/activity show up correctly?"
             "sks person as astronaut" → does the image look like an astronaut?

DINO Score  → "Do the visual details match the reference images?"
             Same face shape? Similar skin tone? Recognizable features?

FaceSim     → "Is this actually the same identity?"
             Face recognition metric — strongest signal for headshots

FID         → "Is the overall image quality/realism good?"
             Not identity-specific, measures photorealism
```

### 🏆 Production Evaluation Protocol

A good headshot system tracks all four and has threshold requirements:

```python
ACCEPT_THRESHOLDS = {
    "face_similarity": 0.50,  # Must match identity
    "clip_score":      0.28,  # Must match prompt
    "dino_score":      0.60,  # Must be visually similar to refs
    "fid":            50.0,   # Must be photorealistic
}
```

## 🏗️ Overall System Design

Putting it all together — how would you design a production personalized headshot system?

```
┌──────────────────────────────────────────────────────────────────────┐
│                    PERSONALIZED HEADSHOT PIPELINE                    │
└──────────────────────────────────────────────────────────────────────┘

   USER UPLOADS 10-20 PHOTOS
           │
           ▼
┌──────────────────────────────┐
│      DATA PIPELINE           │
│  1. Face detection (MTCNN)   │  ← Reject photos with no clear face
│  2. Face alignment           │  ← Normalize position/scale
│  3. Quality checks:          │
│     - Min resolution 512px   │
│     - Blur detection         │  ← Laplacian variance threshold
│     - Face size > 25% frame  │
│     - Frontal-ish pose       │  ← Head pose estimation
│  4. Diversity filter         │  ← Don't want 10 near-identical selfies
│  5. NSFW filter              │
└──────────────┬───────────────┘
               │  5-15 high-quality face crops
               ▼
┌──────────────────────────────┐
│      TRAINING PIPELINE       │
│  Base: SDXL or SD 1.5        │
│  Method: LoRA (rank 8-16)    │
│  Trigger: rare token [V]     │
│  Steps: 1000-1500            │
│  LR: 1e-4 (A_matrix)         │
│      1e-5 (B_matrix)         │
│  PPL: optional but helpful   │
│  Time: ~30 min / A100 GPU    │
└──────────────┬───────────────┘
               │  trained LoRA adapter (50-100MB)
               ▼
┌──────────────────────────────┐
│      INFERENCE PIPELINE      │
│  Load base model + LoRA      │
│  Prompt: "[V] person as      │
│    professional LinkedIn     │
│    headshot, studio lighting"│
│  Steps: 30-50 (DDIM/DPM++)  │
│  CFG scale: 7-9              │
│  Batch: 4-8 images           │
└──────────────┬───────────────┘
               │  candidate images
               ▼
┌──────────────────────────────┐
│      QUALITY ASSESSMENT      │
│  - FaceNet similarity > 0.5  │  ← Identity check
│  - CLIP score > 0.28         │  ← Prompt alignment
│  - DINO score > 0.60         │  ← Visual similarity
│  - NSFW filter               │
│  - Rank and return top-K     │
└──────────────────────────────┘
               │
               ▼
         ✅ FINAL HEADSHOTS
```

### ⚠️ Key Engineering Challenges

1. **Cold start latency**: Training takes 30 min. Solutions: pre-warm GPU pools, async job queues (SQS/Kafka)
2. **GPU cost**: A100/H100 are expensive. Use spot instances + checkpoint saving
3. **Storage at scale**: 1M users × 100MB = 100TB. Use tiered storage (hot/warm/cold)
4. **Identity leakage**: Never cross-contaminate user LoRA adapters
5. **Face diversity in training set**: If all 15 photos are similar selfies, model overfits to one angle
6. **Consent & compliance**: GDPR — users must be able to delete their model

## 📋 Interview Cheat Sheet

### Top Questions and Power Answers

---

**Q: "Explain the three main personalization methods and their trade-offs."**

A: "Textual Inversion learns only a new text embedding (~5KB) but has the weakest identity fidelity. DreamBooth fine-tunes the entire model (2-6GB) and achieves the highest quality using a rare token identifier and prior preservation loss to prevent forgetting. LoRA injects small low-rank matrices (A·B) into attention layers, achieving 95-99% parameter savings versus full fine-tuning while maintaining high quality — it's the production standard today."

---

**Q: "Why does DreamBooth use rare tokens like 'sks' instead of common words?"**

A: "Common words have strong priors from pretraining — using 'john' creates a tug-of-war between 'John = a generic man' and 'John = this specific person'. Random strings that aren't in the vocabulary get split by BPE into multiple tokens, making conditioning messy. Rare single-token identifiers like 'sks' exist in the vocabulary but have almost no learned associations, giving the model a blank slate to associate exclusively with the subject's identity."

---

**Q: "What is catastrophic forgetting in the context of DreamBooth, and how does PPL fix it?"**

A: "After fine-tuning on 15 photos of one person, the model starts associating 'a person' exclusively with that identity — it forgets what generic faces look like. Prior Preservation Loss (PPL) fixes this by co-training on images generated by the original model for 'a person' (called prior images). This regularizes the model to maintain its general understanding of the class while learning the specific subject."

---

**Q: "Walk me through LoRA's math. Why is B initialized to zero?"**

A: "LoRA decomposes weight updates as ΔW = B·A where B is (d_out × r) and A is (r × d_in) with r << d. The full forward pass becomes W_new·x = (W_frozen + (α/r)·B·A)·x. B is initialized to zero so that at the start of training, ΔW = 0 — the LoRA layer behaves exactly like the original frozen layer. This is crucial for stable training: we don't want random noise injected into the model on step 0. A is initialized with Kaiming uniform to break symmetry."

---

**Q: "Why use DINO score instead of just CLIPScore for evaluation?"**

A: "CLIP was trained to align images with text descriptions — it captures semantic similarity (both images show 'a person'), but misses fine visual details. DINO is a self-supervised ViT model that learns patch-level visual features without text supervision, making it sensitive to fine details like face shape, texture, and specific visual patterns. For headshot personalization, you need both: CLIPScore to verify prompt adherence (is the astronaut scene correct?) and DINO/FaceSim to verify identity preservation (is it actually the same person?)."

---

**Q: "How would you handle 1 million users each with a LoRA model at production scale?"**

A: "Use tiered storage: hot storage (SSD/NVMe) for recently active users' LoRAs, warm storage (S3) for users active in last 30 days, cold storage (Glacier) for inactive users. At inference, stream the LoRA adapter from S3 (~100MB in <1s) and merge with the base model in memory. Consider LoRA multiplexing — load multiple user adapters in one GPU batch. For training, use async job queues with spot instance GPU pools, checkpointing every 100 steps. For compliance, implement hard delete that removes the adapter from all storage tiers."

## 🧪 Quiz: Test Your Knowledge!

Five questions to check your understanding. Try to answer before revealing!

---

**Question 1** 🤔

In Textual Inversion, you have a 768-dimensional embedding vector to train. The UNet has 860 million parameters. What percentage of the total model parameters are you actually training?

<details>
<summary>👆 Click to reveal answer</summary>

768 / (860,000,000 + 768) ≈ **0.000089%** — less than one ten-thousandth of a percent! This is why Textual Inversion is so storage-efficient (just that one vector), but also why it has limited expressiveness — one vector can only capture so much about a face.

</details>

---

**Question 2** 🤔

A developer fine-tunes a DreamBooth model on 15 photos of their friend using the token "smith". After training, they try the prompt "a photo of a person" and it keeps generating their friend's face. What went wrong and how do you fix it?

<details>
<summary>👆 Click to reveal answer</summary>

Two problems: (1) **Common word used as identifier** — "smith" is a common English surname with strong prior associations, causing interference. Should use a rare token like "sks" instead. (2) **Missing Prior Preservation Loss** — without PPL, fine-tuning on one person causes the model to associate "person" with that individual (catastrophic forgetting). Fix: add PPL with prior images generated by the original model for "a photo of a person".

</details>

---

**Question 3** 🤔

You're building a production headshot system and need to choose between LoRA rank=4 and rank=16. Your team cares about:
- Storage: 1M users
- Identity fidelity: must be recognizable
- Training cost: A100 time

Which rank do you choose and why?

<details>
<summary>👆 Click to reveal answer</summary>

For most production systems, **rank=8** (or rank=4 if highly cost-constrained) is the right answer.

- **Rank=4**: ~6,144 params per layer → very storage efficient, training ~25% faster than r=16, adequate identity fidelity for most faces
- **Rank=16**: ~24,576 params per layer → 4x more storage/compute, noticeably better identity fidelity for complex cases

At 1M users × 100 layers × 4 bytes:
- r=4: 1M × 6,144 × 100 × 4 ≈ 2.5 TB
- r=16: 1M × 24,576 × 100 × 4 ≈ 10 TB

The practical answer: start with r=4, A/B test quality, upgrade to r=8 if users complain about identity fidelity.

</details>

---

**Question 4** 🤔

Why is B initialized to **zero** in LoRA but A is initialized with Kaiming uniform? What would happen if you initialized both A and B with random values?

<details>
<summary>👆 Click to reveal answer</summary>

**B=0 ensures ΔW = B·A = 0 at initialization** — the LoRA layer starts as a perfect pass-through of the frozen weights. This is critical because:

1. Training starts from a known stable point (the pretrained model's behavior)
2. No random noise is injected into the forward pass on step 0
3. The gradients are well-conditioned from the start

**If both A and B were random:** ΔW = B·A would be a random matrix, immediately corrupting the model's outputs. The loss would start very high, gradients would be unstable, and training would likely diverge or take much longer.

**Why Kaiming for A?** A provides the initial "direction" of updates. Kaiming uniform ensures the variance of activations is preserved through the A projection, preventing vanishing/exploding gradients as training begins.

</details>

---

**Question 5** 🤔

A user submits 20 photos for personalized headshot training. Your quality check rejects 12 of them, leaving only 8. The remaining 8 are all slightly different angles of the same selfie pose. What problems do you anticipate and what would you do?

<details>
<summary>👆 Click to reveal answer</summary>

**Problems:**
1. **Pose overfitting** — model learns the specific selfie angle so strongly that it generates that exact pose even when prompted for different poses ("sitting at a desk" still shows a front-facing close-up)
2. **Limited lighting variation** — identical lighting conditions mean the model won't generalize to different lighting scenarios
3. **Background contamination** — if all 8 photos have the same background, the background becomes part of the "identity signal"

**Solutions:**
1. **Ask for more diverse photos** — add a warning/request for full-body, profile, different expressions
2. **Data augmentation** — apply random crops, flips, color jitter, synthetic lighting changes
3. **Diversity penalty in data selection** — use DINO embeddings to cluster photos and ensure you pick from different clusters (don't want 8 near-identical shots)
4. **Reduce training steps** — with low-diversity data, fewer steps prevent overfitting to the specific pose
5. **Increase PPL weight** — stronger prior preservation to counteract overfitting

</details>